# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abcreativeakshay/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

---

I picked two findings from the FlyRank research paper (*The State of AI-Driven SEO*,
March 2026) and audited each the same way: **where does the label come from, and does
the validation design carry the claim?** Constructive tone intended — both findings
are more careful than most, and my questions are about their edges.

**Finding #1 — "The Anatomy of Growing Content" (marked CONFIRMED).**
Growing pages are reported as 37.6% longer (3.2K vs 2.3K average words) and 20%
younger (184 vs 230 days) than declining pages, on a large sample (74,187 rising vs
45,272 falling).
*Label origin:* the groups are defined by impression trend — the paper's own
definition is a 30d-vs-prev-30d impression change (Up > +10%, Down < −10%). So the
"label" is a measured trend, and word count / age are compared across groups the
outcome itself defined. That is a legitimate descriptive design, and the paper says
so itself ("this remains an observational comparison").
*Does the design carry the claim?* It carries the *descriptive* claim — the two
cohorts do differ, on big samples. It does not by itself carry the action claim
("expand thin pages that already earn impressions"), because the comparison cannot
separate "length helps a page grow" from "pages on growing topics receive more
investment and end up longer". My methodology question: *was word count measured
before the trend window, or across it?* If length grew during the same window the
trend is measured over, part of the gap is mechanical.
*My check on our starter slice (code below):* the direction does **not** replicate
here. In the 30,000-page slice, observed-declining pages average **2,512 words** vs
**1,882** for rising pages, and declining pages are *younger* (236 vs 289 days) —
the opposite sign on both numbers. That doesn't refute the paper (a different,
smaller portfolio; and our `word_count` has structural missingness — one content
type is ~100% missing and filled with 0, which distorts group means). It does show
the finding is portfolio-specific: a directional claim measured on one book of
content has to be re-measured, not assumed, on another.

**Finding #2 — "The Content Performance Curve" (marked CONFIRMED).**
Health score peaks at 61–90 days (33.1), falls off a "decay cliff" at 271–365 days
(14), and rebounds at 365+ (25.1), with the rebound concentrated in refreshed pages.
*Label origin:* the outcome here is not a raw metric — it is FlyRank's own
`health_score`, a composite (impressions 30pts + position 30 + CTR 20 + scroll 20).
The paper discloses this and pairs it with raw search performance, which is the
right practice; but the *shape* of the curve is partly a property of how the
composite weights its parts, so the finding is one step more constructed than
Finding #1.
*Does the design carry the claim?* The age buckets are cross-sectional — different
pages at different ages, not the same pages followed over time — so cohort mix
(what kinds of pages survive to 365+?) can draw part of the curve. The paper itself
flags the key selection effect honestly: the 365+ rebound is concentrated in pages
that were refreshed, i.e. pages someone chose to invest in. My methodology
question: *within the 365+ bucket, how large is the refreshed-vs-unrefreshed gap,
and would the rebound survive comparing only unrefreshed pages?* The actionable
core — review pages before the 9–12-month zone — is decision-support the design
does carry; a per-page lifecycle claim ("content decays at 271 days") would need a
longitudinal design it doesn't have.

**The thread connecting both:** each finding's label is a *derived* quantity (a
trend bucket; a composite score). That is exactly the situation in my own work —
`is_declining_label` is derived from `trend_direction` — and it is why the rest of
this notebook treats label origin as the first audit question, for the paper and
for me alike.


In [1]:
# --- Setup: find the repo root, load the prepared feature vector -------------------
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw" / "content_refresh_anonymized.csv").exists() and _root != _root.parent:
    _root = _root.parent
ROOT = _root
sys.path.insert(0, str(ROOT / "scripts"))

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

SEED = 42  # fixed everywhere in this notebook, same as the reference pipeline

FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
if not FEATURE_PATH.exists():
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "scripts" / "01_prepare_features.py")], check=True)

fv = pd.read_csv(FEATURE_PATH)
y = fv["is_declining_label"].astype(int)
print(f"Feature vector: {len(fv):,} rows x {len(fv.columns)} cols")
print(f"Declining-label base rate (full data): {y.mean():.3f}")

# Check 1 — how clean is the label's origin in OUR slice?
# The paper defines trend from a 30d-vs-prev-30d change (+/-10%). trend_pct is the
# starter slice's sibling of that change; if trend_direction were exactly its
# thresholding, the bands below would be pure. They are not — one more reason
# trend_pct stays banned as a feature (Section 3).
fv["trend_band"] = pd.cut(fv["trend_pct"], [-np.inf, -10, 10, np.inf],
                          labels=["trend_pct < -10", "-10..10", "trend_pct > 10"])
print(pd.crosstab(fv["trend_band"], fv["trend_direction"], normalize="index").round(3))

# Check 2 — Finding #1's direction, re-measured on the starter slice
analogue = fv.groupby("trend_direction").agg(
    pages=("content_id", "size"), avg_words=("word_count", "mean"),
    avg_age_days=("content_age_days", "mean"),
    avg_impressions=("impressions_90d", "mean")).round(1)
print("\nPaper Finding #1 analogue in the starter slice:")
print(analogue)
print("\nPaper (full portfolio): growing = 3.2K words / 184d; declining = 2.3K words / 230d")
print("Starter slice:            declining pages are LONGER and YOUNGER here — sign flips.")
print("Caveat: word_count is structurally missing for one content type (filled 0),")
print("so these means are distorted; the point is 're-measure, don't assume', not 'refuted'.")

Feature vector: 30,000 rows x 52 cols
Declining-label base rate (full data): 0.542
trend_direction  down  flat    new  stable     up
trend_band                                       
trend_pct < -10  0.89  0.00  0.000   0.110  0.000
-10..10          0.00  0.18  0.349   0.471  0.000
trend_pct > 10   0.00  0.00  0.000   0.176  0.824

Paper Finding #1 analogue in the starter slice:
                 pages  avg_words  avg_age_days  avg_impressions
trend_direction                                                 
down             16262     2512.0         236.2           4919.1
flat              1152     2289.0         245.9             20.7
new               2236     2265.0         238.7            164.1
stable            5962     2096.3         295.4           9213.6
up                4388     1881.7         288.5           4716.1

Paper (full portfolio): growing = 3.2K words / 184d; declining = 2.3K words / 230d
Starter slice:            declining pages are LONGER and YOUNGER here — sign fl

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

---

**Before / after, same model, same features, same seed — only the split changes**
(code below). The "before" number is the one a random row split reports; the "after"
is my Week-5 client-holdout.

| Split | P@20 | P@50 | P@100 | ROC AUC | Avg precision |
|---|---|---|---|---|---|
| Random row split (stratified) — *before* | 0.90 | 0.90 | 0.90 | 0.758 | 0.769 |
| Client holdout (6 clients) — *after* | 0.70 | 0.68 | 0.70 | 0.747 | 0.610 |

Base rates: random-split test ≈ 0.542; holdout test = 0.391.

**How I read the gap.** The AUC barely moves (0.758 → 0.747) but Precision@50 drops
by 22 points (0.90 → 0.68). Both facts matter. The ranking *within* a portfolio is
mostly real signal — that is the AUC holding up. But the very top of the list was
inflated by client identity: under a row split, the model sees thousands of pages
from every client in training, learns each client's character, and its top 50 is
then partly "pages from clients it knows decline a lot". On clients it has never
seen, that shortcut disappears and P@50 falls to the honest 0.68. The gap between
the two numbers *is* the measurement of how much memorisation the random split was
hiding — which is why every headline number in ML-07/ML-08 is the after number.


In [2]:
def client_holdout(frame, target, seed=SEED, test_share=0.2):
    """Same split as scripts/03_train_model.py: hold out ~20% of CLIENTS (seed 42),
    so no client's pages appear in both train and test."""
    clients = frame["client_id"].fillna("unknown").astype(str).drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(clients)
    n_test = max(1, int(round(len(shuffled) * test_share)))
    test_clients = set(shuffled[:n_test])
    test_mask = frame["client_id"].astype(str).isin(test_clients).to_numpy()
    return np.arange(len(frame))[~test_mask], np.arange(len(frame))[test_mask], test_clients

def build_feature_matrix(frame, extra_flags=()):
    """Same encoding as scripts/03_train_model.py: pipeline numeric features (filled 0)
    + one-hot categoricals. Optionally appends engineered flag columns."""
    numeric = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
    cats = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]
    num = frame[numeric].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = pd.get_dummies(frame[cats].fillna("unknown").astype(str),
                         prefix=cats, dummy_na=False, dtype=float)
    X = pd.concat([num.reset_index(drop=True), cat.reset_index(drop=True)], axis=1)
    for flag in extra_flags:
        X[flag] = frame[flag].to_numpy()
    return X

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

def make_rf():
    return RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
                                  min_samples_leaf=25, n_estimators=200,
                                  n_jobs=-1, random_state=SEED)

def row(name, y_true, scores):
    return {"split": name,
            "P@20": round(precision_at_k(y_true, scores, 20), 3),
            "P@50": round(precision_at_k(y_true, scores, 50), 3),
            "P@100": round(precision_at_k(y_true, scores, 100), 3),
            "ROC AUC": round(roc_auc_score(y_true, scores), 3),
            "Avg precision": round(average_precision_score(y_true, scores), 3),
            "test base rate": round(float(np.mean(y_true)), 3)}

X = build_feature_matrix(fv)
rows = []
# BEFORE: random row split (stratified) — the flattering number
r_tr, r_te = train_test_split(np.arange(len(fv)), test_size=0.2, random_state=SEED, stratify=y)
m = make_rf().fit(X.iloc[r_tr], y.iloc[r_tr])
rows.append(row("random row split (before)", y.iloc[r_te], m.predict_proba(X.iloc[r_te])[:, 1]))
# AFTER: grouped client holdout — the honest number
g_tr, g_te, _ = client_holdout(fv, y)
m = make_rf().fit(X.iloc[g_tr], y.iloc[g_tr])
rows.append(row("client holdout (after)", y.iloc[g_te], m.predict_proba(X.iloc[g_te])[:, 1]))
print(pd.DataFrame(rows).to_string(index=False))

                    split  P@20  P@50  P@100  ROC AUC  Avg precision  test base rate
random row split (before)   0.9  0.90    0.9    0.758          0.769           0.542
   client holdout (after)   0.7  0.68    0.7    0.747          0.610           0.391


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

---

I ran the Week-3 hunt on my final ML-08 feature set (the pipeline's 18 numeric +
8 categorical features, plus the three experiment flags). Verdicts, in the
taxonomy's order:

**1. Label-derived features — clean, and I can prove the tripwire works.** Neither
`trend_direction` (the label's parent) nor `trend_pct` (its sibling) is in the
feature matrix — asserted in code. As the control experiment, I deliberately added
`trend_pct` for one run: **P@50 and AUC both went to exactly 1.000**. A perfect
score is not a result, it is a confession — and seeing the harness produce it on
demand confirms the honest 0.68 is measuring something real. The leaky run is
labelled, quarantined to this audit, and never used anywhere else.

**2. Product / decision-derived features — none exist to use.** The starter data
intentionally ships no FlyRank decision flags (`health_score`, `needs_ctr_fix`, …),
so there is nothing circular to learn from. Verified in code: no such columns are
present.

**3. IDs — grouping only.** `content_id` / `client_id` are used for the split and
for joining queues, never as features (asserted). The split itself is the grouped
one from ML-08.

**4. Windows — the one real caveat, disclosed.** Every feature is a trailing-90-day
aggregate or a page attribute at the snapshot; the label's trend is computed
*inside* that same trailing window (last 30d vs previous 30d). So my features are
not strictly "before" the label window the way a production past→future design
would require — the starter snapshot simply has no future window to label from.
Consequences I accept and state: (a) my numbers describe *measured association on
held-out clients*, i.e. decision-support for review order, not forecasts of what
will decline next; (b) the warehouse release exists precisely to fix this — its
`*_prev30`-style columns are the legal features for a last-30-days label. Anyone
quoting ML-08 without this paragraph is quoting it wrong.

**5. Population selection — disclosed.** The feature vector keeps only pages with
`impressions_90d > 0` and `content_age_days >= 90` (step 01's filter). That
population is defined on observation-window quantities, not on the outcome, and it
means my claims cover *established, visible* pages only — new or invisible pages
are out of scope by construction (consistent with the false-negative pattern in
ML-08).

**6. The experiment flags — not leakage, but a concentration warning.**
`has_clicks`, `has_ai_sessions`, `measurable_opportunity` are observable at
scoring time and are not computed from the label (their label rates are unremarkable:
0.578 vs 0.496; 0.553 vs 0.541; 0.598 vs 0.389). The audit concern is different:
the flags variant's holdout top 50 is 49/50 pages from a single client, so its
P@50 = 0.84 is one portfolio's story until a wider holdout says otherwise.


In [3]:
# --- Leakage audit: tripwires, the deliberate-leak control, and the flags check --------
feature_cols = list(X.columns)
assert "trend_direction" not in feature_cols and "trend_pct" not in feature_cols
assert "content_id" not in feature_cols and "client_id" not in feature_cols
print("Feature matrix contains no label-derived columns and no IDs (asserted).")

product_flags = [c for c in fv.columns if c in
                 {"health_score", "needs_ctr_fix", "is_quick_win", "is_zombie"}]
print("Product decision flags present in the data:", product_flags or "none (not shipped)")

# CONTROL EXPERIMENT (labelled, quarantined): add the label's sibling and watch it confess
X_leak = X.copy(); X_leak["trend_pct"] = fv["trend_pct"].to_numpy()
m_leak = make_rf().fit(X_leak.iloc[g_tr], y.iloc[g_tr])
p_leak = m_leak.predict_proba(X_leak.iloc[g_te])[:, 1]
print(f"\nDELIBERATE LEAK (trend_pct added): P@50 = {precision_at_k(y.iloc[g_te], p_leak, 50):.3f}, "
      f"AUC = {roc_auc_score(y.iloc[g_te], p_leak):.3f}  <- a perfect score is a confession,")
print("not a result. The honest model (previous section) stays at P@50 = 0.68.")

# The experiment flags: observable, not label-derived — but check concentration
print("\nFlag label-rates (weak individually, so not label proxies):")
for c in ["has_clicks", "has_ai_sessions", "measurable_opportunity"]:
    print(f"  {c}: label rate when 1 = {y[fv[c] == 1].mean():.3f}, "
          f"when 0 = {y[fv[c] == 0].mean():.3f}, share = {(fv[c] == 1).mean():.2f}")
X_flags = build_feature_matrix(fv, extra_flags=["has_clicks", "has_ai_sessions",
                                                "measurable_opportunity"])
m_flags = make_rf().fit(X_flags.iloc[g_tr], y.iloc[g_tr])
pred = fv.iloc[g_te].copy()
pred["prob"] = m_flags.predict_proba(X_flags.iloc[g_te])[:, 1]
top50 = pred.sort_values("prob", ascending=False).head(50)
print("\nFlags-variant top-50 client mix:", top50["client_id"].value_counts().to_dict(),
      " <- concentration warning, not leakage")

Feature matrix contains no label-derived columns and no IDs (asserted).
Product decision flags present in the data: none (not shipped)



DELIBERATE LEAK (trend_pct added): P@50 = 1.000, AUC = 1.000  <- a perfect score is a confession,
not a result. The honest model (previous section) stays at P@50 = 0.68.

Flag label-rates (weak individually, so not label proxies):
  has_clicks: label rate when 1 = 0.578, when 0 = 0.496, share = 0.56
  has_ai_sessions: label rate when 1 = 0.553, when 0 = 0.541, share = 0.06
  measurable_opportunity: label rate when 1 = 0.598, when 0 = 0.389, share = 0.73



Flags-variant top-50 client mix: {'client_f74efabef1': 49, 'client_4fc82b26ae': 1}  <- concentration warning, not leakage


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

---

**My boldest sentence from ML-08, as I first drafted it:**

> "Adding three simple availability flags lifts the model to 0.84 Precision@50 — it
> finds the right pages five times better than the hand rule, so the flags version
> should be the production ranker."

**Why it doesn't hold as written:** "0.84" was measured on one six-client holdout
whose top 50 is 49/50 pages from a single client; "five times better" quietly
compares against the reference rule's weakest slice number; and "should be the
production ranker" is a deployment claim no measurement here carries.

**The rewrite — same finding, safe language:**

> "On six held-out clients (2,325 pages, observed declining base rate 0.391), adding
> three observable availability flags (`has_clicks`, `has_ai_sessions`,
> `measurable_opportunity`) to the random forest changed the measured Precision@50
> from 0.68 to 0.84, directionally above both the model without flags (0.68) and the
> hand-rule baselines (0.24–0.48) on the same slice. The flags variant's top 50 was
> concentrated in a single held-out client (49 of 50 pages), so I treat the size of
> the gain as directional and unverified beyond this slice. The result is
> decision-support for the order of a human review queue; whether refreshing the
> flagged pages changes their outcomes was not measured."

Every number in that paragraph is printed by the code below and saved with the
ML-08 receipt — a claim whose receipts are in the repo, not in my memory.


In [4]:
# --- Receipts for the rewritten claim ---------------------------------------------------
receipt = {
    "assignment": "ML-09",
    "seed": SEED,
    "honest_split": {"name": "client_holdout", "test_rows": int(len(g_te)),
                     "test_base_rate": float(y.iloc[g_te].mean())},
    "random_forest": {"random_row_split": rows[0], "client_holdout": rows[1]},
    "deliberate_leak_control": {
        "added_feature": "trend_pct",
        "precision_at_50": float(precision_at_k(y.iloc[g_te], p_leak, 50)),
        "roc_auc": float(roc_auc_score(y.iloc[g_te], p_leak)),
        "verdict": "perfect score = leakage confession; feature banned everywhere else"},
    "flags_variant_top50_client_mix": {k: int(v) for k, v in
                                       top50["client_id"].value_counts().items()},
}
out_path = ROOT / "work" / "outputs" / "w06_validation_audit.json"
out_path.write_text(json.dumps(receipt, indent=2))
print(json.dumps(receipt, indent=2))
print(f"\nSaved receipt: {out_path.relative_to(ROOT)}")

{
  "assignment": "ML-09",
  "seed": 42,
  "honest_split": {
    "name": "client_holdout",
    "test_rows": 2325,
    "test_base_rate": 0.3909677419354839
  },
  "random_forest": {
    "random_row_split": {
      "split": "random row split (before)",
      "P@20": 0.9,
      "P@50": 0.9,
      "P@100": 0.9,
      "ROC AUC": 0.758,
      "Avg precision": 0.769,
      "test base rate": 0.542
    },
    "client_holdout": {
      "split": "client holdout (after)",
      "P@20": 0.7,
      "P@50": 0.68,
      "P@100": 0.7,
      "ROC AUC": 0.747,
      "Avg precision": 0.61,
      "test base rate": 0.391
    }
  },
  "deliberate_leak_control": {
    "added_feature": "trend_pct",
    "precision_at_50": 1.0,
    "roc_auc": 1.0,
    "verdict": "perfect score = leakage confession; feature banned everywhere else"
  },
  "flags_variant_top50_client_mix": {
    "client_f74efabef1": 49,
    "client_4fc82b26ae": 1
  }
}

Saved receipt: work/outputs/w06_validation_audit.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

*Final box left unticked on purpose: the notebook is complete and executed in `work/notebooks/`; committing it to my repo is my own submission step.*